# Week 7 - Document Question Answering System (RAG) - v2
### CEI'26 Data Science Internship - Celebal Technologies

**Assignment:** Build a Retrieval-Augmented Generation (RAG) system to answer questions from custom documents.

---

**What's different in this version:**
The first version used `flan-t5-base` as the generator, which is a very small (250M param) model and gave weak/incomplete answers. This version swaps it for `Qwen2.5-1.5B-Instruct`, a proper instruction-tuned chat model, and cleans up the PDF text extraction + prompt so the answers are actually coherent and grounded in the document.

Pipeline: PDF -> clean text -> chunk -> embed (sentence-transformers) -> store in FAISS -> retrieve top-k chunks for a query -> generate an answer with an instruction-tuned LLM using only that retrieved context.

## 1. Setup - Installing Libraries

In [3]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate langchain-text-splitters

## 2. Imports

In [4]:
import os
import re
import textwrap
import numpy as np
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_text_splitters import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


## 3. Loading the Document

Using a custom PDF as the knowledge source (RAG is meant for private/custom data anyway, as mentioned in the assignment doc).

On Kaggle/Colab -> upload your own PDF (notes, resume, research paper, etc.) and update `PDF_PATH` below.

In [5]:

# from google.colab import files
# uploaded = files.upload()
# PDF_PATH = list(uploaded.keys())[0]

PDF_PATH = "Intro to Cognitive computing.pdf"

if not os.path.exists(PDF_PATH):
    print(f"'{PDF_PATH}' not found. Please upload a PDF and update PDF_PATH.")
else:
    print(f"Found file: {PDF_PATH}")

Found file: Intro to Cognitive computing.pdf


In [6]:
def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
    return full_text

def clean_text(text):
    # PDFs often extract with broken line breaks and extra spaces - clean that up
    text = re.sub(r"-\n", "", text)            # join hyphenated words split across lines
    text = re.sub(r"[ \t]+", " ", text)          # collapse repeated spaces/tabs
    text = re.sub(r"\n{2,}", "\n\n", text)       # collapse excess blank lines
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)  # join single line breaks into flowing sentences
    return text.strip()

raw_text = load_pdf_text(PDF_PATH) if os.path.exists(PDF_PATH) else ""
raw_text = clean_text(raw_text)
print("Total characters extracted:", len(raw_text))
print(raw_text[:500])

Total characters extracted: 2191
Cognitive  Computing AI contains everything: rule-based  systems, optimization, search,  expert systems, robotics, etc. ML is specifically about models  improving with data. Cognitive Computing uses AI +  ML + NLP + reasoning to simulate  human cognition. Quick Figure-Wise Definitions Term Figure-wise Summary Core Capability AI Umbrella technology for  smart machines Reasoning, decisionmaking ML Subset of AI that learns  from data Pattern recognition Cognitive Computing Uses ML + AI + context + 


## 4. Text Chunking

Splitting the cleaned text into overlapping chunks so retrieval can find focused, relevant pieces instead of matching against the whole document at once.

Using a chunk size of 800 characters with 120 character overlap - slightly larger than v1 so each chunk carries enough context for the LLM to actually answer from.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(raw_text) if raw_text else []
chunks = [c.strip() for c in chunks if len(c.strip()) > 20]  # drop tiny/empty chunks
print(f"Number of chunks created: {len(chunks)}")
if chunks:
    print("\nSample chunk:\n")
    print(chunks[0])

Number of chunks created: 4

Sample chunk:

Cognitive  Computing AI contains everything: rule-based  systems, optimization, search,  expert systems, robotics, etc. ML is specifically about models  improving with data. Cognitive Computing uses AI +  ML + NLP + reasoning to simulate  human cognition. Quick Figure-Wise Definitions Term Figure-wise Summary Core Capability AI Umbrella technology for  smart machines Reasoning, decisionmaking ML Subset of AI that learns  from data Pattern recognition Cognitive Computing Uses ML + AI + context +  language to think like  humans Human-like  understanding Example Breakdown (with Tasks) Feature AI ML Cognitive Computing Learn from data Understand natural language Reason & draw insights Adapt through experience Mimic human thought


## 5. Creating Embeddings

Using `all-MiniLM-L6-v2` from sentence-transformers - small, fast, free (runs on CPU), and good enough for semantic retrieval on a document-sized corpus.

In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True) if chunks else np.array([])
print("Embedding matrix shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (4, 384)


## 6. Building the Vector Database (FAISS)

Storing chunk embeddings in a FAISS index for fast similarity search. Using `IndexFlatL2` since the dataset is small - for a much bigger corpus you'd switch to an approximate index like IVF or HNSW.

In [9]:
dimension = chunk_embeddings.shape[1] if len(chunk_embeddings) else 384
index = faiss.IndexFlatL2(dimension)

if len(chunk_embeddings):
    index.add(chunk_embeddings)

print("Total vectors stored in FAISS index:", index.ntotal)

Total vectors stored in FAISS index: 4


## 7. Query Processing + Retrieval

Convert the user's question into an embedding using the same model, then search the FAISS index for the top-k most similar chunks.

In [10]:
def retrieve_context(query, top_k=4):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0] if i < len(chunks)]
    return retrieved_chunks

# quick test
if chunks:
    test_query = "What is the main idea of the document?"
    retrieved = retrieve_context(test_query, top_k=4)
    for i, r in enumerate(retrieved):
        print(f"--- Retrieved Chunk {i+1} ---")
        print(textwrap.fill(r, 100))
        print()

--- Retrieved Chunk 1 ---
• Predicted that by 2000, a machine might have a 30% chance of fooling a lay person for 5  minutes •
Anticipated all major arguments against AI in following 50 years • Suggested major components of AI:
knowledge, reasoning, language understanding, learning

--- Retrieved Chunk 2 ---
Cognitive  Computing AI contains everything: rule-based  systems, optimization, search,  expert
systems, robotics, etc. ML is specifically about models  improving with data. Cognitive Computing
uses AI +  ML + NLP + reasoning to simulate  human cognition. Quick Figure-Wise Definitions Term
Figure-wise Summary Core Capability AI Umbrella technology for  smart machines Reasoning,
decisionmaking ML Subset of AI that learns  from data Pattern recognition Cognitive Computing Uses
ML + AI + context +  language to think like  humans Human-like  understanding Example Breakdown
(with Tasks) Feature AI ML Cognitive Computing Learn from data Understand natural language Reason &
draw insights 

## 8. Answer Generation

Using `Qwen2.5-1.5B-Instruct` as the generator instead of flan-t5-base. It's still free and runs locally (no API key), but being a proper instruction-tuned chat model it gives much more coherent, complete answers instead of the disconnected/echoing outputs flan-t5-base was producing.

Uses the model's chat template (system + user message) so it correctly follows the "answer only from context" instruction. Runs on GPU if available, otherwise CPU (a bit slower but still works fine for a handful of queries).

In [11]:
gen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

def generate_answer(query, top_k=4):
    retrieved_chunks = retrieve_context(query, top_k=top_k)
    context = "\n\n".join(retrieved_chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions strictly using the "
                "provided document context. If the answer is not present in the context, "
                "say you don't know instead of making something up. Keep answers clear and concise."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]

    prompt = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)

    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=gen_tokenizer.eos_token_id
    )

    # only decode the newly generated tokens, not the prompt itself
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    result = gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return result, retrieved_chunks

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## 9. Putting It All Together - RAG Pipeline

Full flow: Question -> Retrieve relevant chunks -> Augment prompt with context -> Generate grounded answer.

In [12]:
def rag_pipeline(query, top_k=4, show_context=True):
    answer, retrieved_chunks = generate_answer(query, top_k=top_k)

    print("QUESTION:", query)
    print("\nANSWER:", answer)

    if show_context:
        print("\n--- Retrieved Context Used ---")
        for i, c in enumerate(retrieved_chunks):
            print(f"[{i+1}] {textwrap.fill(c, 100)}\n")

    return answer

### Example Queries

Testing the pipeline with a few sample questions on the uploaded document.

In [13]:
if chunks:
    sample_questions = [
        "What is the main idea of the document?",
        "Summarize the key points in a few lines.",
    ]

    for q in sample_questions:
        rag_pipeline(q)
        print("=" * 100)
else:
    print("No document loaded yet - upload a PDF and re-run the earlier cells.")

QUESTION: What is the main idea of the document?

ANSWER: The main idea of the document is to introduce and explain different concepts related to artificial intelligence and cognitive computing, including their definitions, core capabilities, applications, and distinctions between them. The text aims to provide an overview of how these technologies work and what they can achieve, focusing on the role of machine learning within this broader framework.

--- Retrieved Context Used ---
[1] • Predicted that by 2000, a machine might have a 30% chance of fooling a lay person for 5  minutes •
Anticipated all major arguments against AI in following 50 years • Suggested major components of AI:
knowledge, reasoning, language understanding, learning

[2] Cognitive  Computing AI contains everything: rule-based  systems, optimization, search,  expert
systems, robotics, etc. ML is specifically about models  improving with data. Cognitive Computing
uses AI +  ML + NLP + reasoning to simulate  human co

### Try Your Own Question

In [14]:
# Change this to any question about your uploaded document
my_question = "What is AI?"

if chunks:
    rag_pipeline(my_question)

QUESTION: What is AI?

ANSWER: AI stands for Artificial Intelligence. It refers to the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction.

--- Retrieved Context Used ---
[1] AI vs Cognitive computing Introduction to AI Artificial Intelligence (AI) Cognitive Computing Makes
machines intelligent Mimics human thinking process Performs tasks like humans Thinks, reasons, and
learns like humans Works mainly on structured data Handles unstructured and uncertain data Focuses
on automation Focuses on decision support Can replace human effort Assists humans, not replaces
Rule-based or model-based systems Context-aware and adaptive systems Example: Chess programs, spam
filters Example: IBM Watson

[2] Simple Real-World Analogy (Figures) • AI = Entire Factory ML = Machines that learn to improve
product quality Cognitive Computing = Machines that understand customers & reasons about  quality
like humans What is AI? Views of AI fall in

## 10. Improvements Tried / Possible Next Steps

- **Generator swap:** Moved from `flan-t5-base` (weak, 250M params, was just echoing the prompt back) to `Qwen2.5-1.5B-Instruct`, a proper instruction-tuned chat model - this alone fixed most of the answer quality issues
- **Text cleaning:** Added a cleanup step for PDF-extracted text (hyphenation, broken line breaks) before chunking, since messy text was hurting both retrieval and generation
- **Chunking:** Increased chunk size (500 -> 800 chars) with more overlap so each chunk carries enough context on its own
- **Further ideas:** Hybrid search (BM25 + vector), a cross-encoder re-ranker on top of retrieved chunks, or an even larger model (7B+) if GPU memory allows

## 11. Key Learnings

- Generator model choice matters a lot in RAG - a weak/wrong-task model can make an otherwise correct retrieval pipeline look completely broken
- Using a causal instruction-tuned model with a proper chat template (system + user roles) gives much better-controlled outputs than a raw text2text pipeline
- Cleaning PDF-extracted text before chunking noticeably improves both retrieval accuracy and final answer quality
- How the 3 core pieces of RAG (Retrieval, Augmentation, Generation) fit together in a working pipeline
- Grounding an LLM's answer in retrieved context reduces hallucination compared to relying on the model's internal knowledge alone

## 12. Conclusion

This notebook implements a complete, working RAG pipeline - from raw PDF ingestion to grounded answer generation - entirely with free, local tools (sentence-transformers + FAISS + Qwen2.5-1.5B-Instruct). Compared to the first version, the answers are now coherent and properly grounded in the retrieved context instead of just echoing the prompt back. It can be pointed at any custom PDF (notes, resume, research paper) and answer questions specifically about that document.